# Batch Normalization（批归一化）

Batch Normalization（BN）在每个通道上标准化激活值，再通过可学习的缩放和偏移恢复表达能力。它有助于稳定训练过程，并允许模型使用更合适的学习率。

对输入 $X\in\mathbb{R}^{N\times C\times H\times W}$，训练时按通道在 $N,H,W$ 维度计算：

$$\mu_c=\frac{1}{M}\sum_{i=1}^{M}x_{i,c},\qquad \sigma_c^2=\frac{1}{M}\sum_{i=1}^{M}(x_{i,c}-\mu_c)^2,\quad M=N\cdot H\cdot W.$$

归一化与仿射变换为：

$$\hat{x}_{i,c}=\frac{x_{i,c}-\mu_c}{\sqrt{\sigma_c^2+\varepsilon}},\qquad y_{i,c}=\gamma_c\hat{x}_{i,c}+\beta_c.$$

训练期间还以动量 $\alpha$ 更新运行统计量 $r\leftarrow(1-\alpha)r+\alpha r_{batch}$；推理时使用这些运行均值和方差。

In [ ]:
import torch
import torch.nn as nn

class MyBatchNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super(MyBatchNorm2d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # 可学习的每通道仿射参数，形状都是 [C]；forward 中会扩展为 [1, C, 1, 1]。
        self.weight = nn.Parameter(torch.ones(num_features))  # gamma: [C]
        self.bias = nn.Parameter(torch.zeros(num_features))   # beta:  [C]
        
        # 运行统计量不是参数，但需随模型保存、加载与迁移设备；形状均为 [C]。
        self.register_buffer('running_mean', torch.zeros(num_features))  # [C]
        self.register_buffer('running_var', torch.ones(num_features))    # [C]
        
    def forward(self, x):
        # x: [N, C, H, W]；所有统计量均按通道 C 独立计算。
        if self.training:
            # 1. 在 N、H、W 上归约，保留 C： [N, C, H, W] -> [C]。
            mean = x.mean(dim=(0, 2, 3))  # [C]
            var = x.var(dim=(0, 2, 3), unbiased=False)  # [C]，有偏方差
            
            # 2. [C] 与 [C] 逐元素更新；no_grad 防止统计量进入反向传播图。
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            # 推理模式：直接取累计的 [C] 运行均值与方差。
            mean = self.running_mean
            var = self.running_var
            
        # 3. [C] -> [1, C, 1, 1]，从而可广播到输入的 [N, C, H, W]。
        x_hat = (x - mean.view(1, -1, 1, 1)) / torch.sqrt(var.view(1, -1, 1, 1) + self.eps)  # [N, C, H, W]
        
        # 4. gamma、beta 同样扩展为 [1, C, 1, 1]；输出形状不变。
        out = x_hat * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)  # [N, C, H, W]
        return out

In [ ]:
import torch
import torch.nn as nn

class MatrixBatchNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        """
        num_features: 特征维度 D (即输入矩阵的最后一维)
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # 可学习参数 (形状为 [D])
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        
        # 滑动平均统计量 (形状为 [D])
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        
    def forward(self, x):
        """
        支持输入形状: 
        - [N, D] (全连接/特征矩阵)
        - [B, L, D] (序列/Transformer特征)
        """
        assert x.dim() in [2, 3], "仅支持 2D 或 3D 输入"
        
        # 1. 确定需要求均值的维度 (除了最后一维 D，其他维度都要归约)
        # 对于 [N, D]，归约 dim=0
        # 对于 [B, L, D]，归约 dim=(0, 1)
        reduce_dims = tuple(range(x.dim() - 1))  # 例如 dim=2 -> (0,), dim=3 -> (0, 1)
        
        if self.training:
            # 2. 计算当前 batch 的均值和方差 (形状为 [D])
            mean = x.mean(dim=reduce_dims) 
            var = x.var(dim=reduce_dims, unbiased=False) # 使用有偏方差
            
            # 3. 更新滑动平均
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            # 推理模式：使用累积的滑动平均
            mean = self.running_mean
            var = self.running_var
            
        # 4. 维度对齐 (利用广播机制)
        # 动态构造 view 形状，例如:
        # 输入 [N, D] -> view_shape = [1, D]
        # 输入 [B, L, D] -> view_shape = [1, 1, D]
        view_shape = [1] * (x.dim() - 1) + [self.num_features]
        
        mean = mean.view(view_shape)
        var = var.view(view_shape)
        weight = self.weight.view(view_shape)
        bias = self.bias.view(view_shape)
        
        # 5. 归一化 + 仿射变换
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        out = x_hat * weight + bias
        
        return out